Procesamiento de Lenguaje Natural  
Proyecto 2 - Chatbot Generador de Tareas para Jira  
Autores: Ing. González Daniel & Ing. Guerra Michael  
Profesor: Dr. Luis Roberto Garcia  

# Librerias

In [1]:
import os, base64, requests
import requests
import anthropic
import json
from typing import Any, Dict, List, Optional
from getpass import getpass
from dotenv import load_dotenv

# Credenciales

In [2]:
load_dotenv()

if "JIRA_BASE" not in os.environ:
    os.environ["JIRA_BASE"] = "https://proyectoX.atlassian.net"

os.environ.pop("JIRA_PAT", None)

if "JIRA_EMAIL" not in os.environ:
    os.environ["JIRA_EMAIL"] = input("JIRA_EMAIL: ").strip()
if "JIRA_TOKEN" not in os.environ:
    os.environ["JIRA_TOKEN"] = getpass("JIRA_TOKEN (API token): ").strip()
    
print("✓ Jira configurado con email+token")
print(f"  Base URL: {os.environ['JIRA_BASE']}")
print(f"  Email: {os.environ['JIRA_EMAIL']}")


✓ Jira configurado con email+token
  Base URL: https://dg-mcia.atlassian.net
  Email: dgonzalez16@alumnos.uaq.mx


In [3]:
def _auth_headers():
    email, token = os.getenv("JIRA_EMAIL"), os.getenv("JIRA_TOKEN")
    if not (email and token):
        raise RuntimeError("Faltan JIRA_EMAIL o JIRA_TOKEN")
    b = base64.b64encode(f"{email}:{token}".encode()).decode()
    return {"Authorization": f"Basic {b}"}

def _base():
    base = os.getenv("JIRA_BASE")
    if not base:
        raise RuntimeError("Falta JIRA_BASE")
    return base.rstrip('/')

# Servidor + Herramientas

In [4]:
def to_adf(text: str) -> dict:
    return {
        "type": "doc",
        "version": 1,
        "content": [
            {"type": "paragraph", "content": [{"type": "text", "text": text}]}
        ]
    }

In [5]:
class ServidorJira:
    """
    Herramientas MCP-style para Jira:
      - create_issue(projectKey, summary, description?, issueType?, assignee?, labels?, customFields?)
      - create_subtask(parentKey, summary, description?, subtaskTypeName?, subtaskTypeId?, assignee?, labels?, customFields?)
      - search_issues(jql, maxResults?)
      - add_comment(issueKey, comment)
      - list_transitions(issueKey)
      - transition_issue(issueKey, transitionId | transitionName, fields?)
    """
    def __init__(self):
        self.tools = [
            {
                "name": "create_issue",
                "description": "Crea un issue en Jira",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "projectKey": {"type": "string"},
                        "summary": {"type": "string"},
                        "description": {"type": "string"},
                        "issueType": {"type": "string", "default": "Task"},
                        "assignee": {"type": "string", "description": "accountId en Cloud"},
                        "labels": {"type": "array", "items": {"type": "string"}},
                        "customFields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["projectKey", "summary"]
                }
            },
            {
                "name": "create_subtask",
                "description": "Crea una sub-tarea bajo un issue padre",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "parentKey": {"type": "string"},
                        "summary": {"type": "string"},
                        "description": {"type": "string"},
                        "subtaskTypeName": {"type": "string", "default": "Sub-task"},
                        "subtaskTypeId": {"type": "string"},
                        "assignee": {"type": "string"},
                        "labels": {"type": "array", "items": {"type": "string"}},
                        "customFields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["parentKey", "summary"]
                }
            },
            {
                "name": "search_issues",
                "description": "Busca issues por JQL",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "jql": {"type": "string"},
                        "maxResults": {"type": "integer", "default": 10},
                        "fields": {
                            "type": "array",
                            "items": {"type": "string"},
                            "default": ["summary", "status", "assignee", "labels", "issuetype"]
                        }
                    },
                    "required": ["jql"]
                }
            },
            {
                "name": "add_comment",
                "description": "Agrega un comentario a un issue",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"},
                        "comment": {"type": "string"}
                    },
                    "required": ["issueKey", "comment"]
                }
            },
            {
                "name": "list_transitions",
                "description": "Lista transiciones disponibles para un issue (nombre, id, destino)",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"}
                    },
                    "required": ["issueKey"]
                }
            },
            {
                "name": "transition_issue",
                "description": "Cambia el estado de un issue (por id o nombre de transición)",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"},
                        "transitionId": {"type": "string"},
                        "transitionName": {"type": "string"},
                        "fields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["issueKey"]
                }
            },
            {
                "name": "get_issue",
                "description": "Obtiene campos de un issue por su clave",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"},
                        "fields": {
                            "type": "array",
                            "items": {"type": "string"},
                            "default": ["summary", "status", "assignee", "labels", "issuetype"]
                        },
                        "expand": {"type": "string"}
                    },
                    "required": ["issueKey"]
                }
            }
        ]

    def get_tools(self):
        return self.tools

    def ejecutar_herramienta(self, name: str, arguments: dict) -> str:
        try:
            if name == "create_issue":
                return self._create_issue(**arguments)
            if name == "create_subtask":
                return self._create_subtask(**arguments)
            if name == "search_issues":
                return self._search_issues(**arguments)
            if name == "add_comment":
                return self._add_comment(**arguments)
            if name == "list_transitions":
                return self._list_transitions(**arguments)
            if name == "transition_issue":
                return self._transition_issue(**arguments)
            if name == "get_issue":
                return self._get_issue(**arguments)
            return f"Herramienta '{name}' no encontrada"
        except TypeError as te:
            return f"❌ Parámetros inválidos: {te}"
        except requests.HTTPError as he:
            try:
                return f"❌ HTTP {he.response.status_code}: {he.response.json()}"
            except Exception:
                return f"❌ HTTP {he.response.status_code}: {he.response.text}"
        except Exception as e:
            return f"❌ Error: {e}"

    # ---------------- Implementaciones ----------------
    def _create_issue(self,
                      projectKey: str, summary: str, description: str = "",
                      issueType: str = "Task", assignee: Optional[str] = None,
                      labels: Optional[List[str]] = None,
                      customFields: Optional[Dict[str, object]] = None) -> str:
        url = f"{_base()}/rest/api/3/issue"
        fields = {
            "project": {"key": projectKey},
            "summary": summary,
            "description": to_adf(description),
            "issuetype": {"name": issueType}
        }
        if assignee:
            fields["assignee"] = {"id": assignee} 
        if labels:
            fields["labels"] = labels
        if customFields:
            fields.update(customFields)

        r = requests.post(url, json={"fields": fields},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        key = r.json().get("key")
        return f"✓ Creado {key} → {_base()}/browse/{key}"

    def _create_subtask(self,
                        parentKey: str, summary: str, description: str = "",
                        subtaskTypeName: str = "Subtask", subtaskTypeId: Optional[str] = None,
                        assignee: Optional[str] = None, labels: Optional[List[str]] = None,
                        customFields: Optional[Dict[str, object]] = None) -> str:
        url = f"{_base()}/rest/api/3/issue"
        issuetype = {"id": subtaskTypeId} if subtaskTypeId else {"name": subtaskTypeName}
        fields = {
            "parent": {"key": parentKey},
            "summary": summary,
            "description": to_adf(description),
            "issuetype": issuetype
        }
        if assignee:
            fields["assignee"] = {"id": assignee}
        if labels:
            fields["labels"] = labels
        if customFields:
            fields.update(customFields)

        r = requests.post(url, json={"fields": fields},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        key = r.json().get("key")
        return f"✓ Sub-tarea {key} bajo {parentKey} → {_base()}/browse/{key}"

    def _search_issues(self, jql: str, maxResults: int = 10,
                   fields: Optional[List[str]] = None,
                   nextPageToken: Optional[str] = None,
                   expand: Optional[str] = None) -> str:
        base = _base()
        headers = {"Accept": "application/json", **_auth_headers()}

        params = {
            "jql": jql,
            "maxResults": maxResults,
        }
        if fields:
            
            params["fields"] = ",".join(fields)
        if nextPageToken:
            params["nextPageToken"] = nextPageToken
        if expand:
            params["expand"] = expand

        url = f"{base}/rest/api/3/search/jql"
        r = requests.get(url, headers=headers, params=params)

        if r.status_code == 405 or r.status_code == 400:
            body = {
                "jql": jql,
                "maxResults": maxResults,
            }
            if fields:
                body["fields"] = fields
            if nextPageToken:
                body["nextPageToken"] = nextPageToken
            if expand:
                body["expand"] = expand
            r = requests.post(url, headers={"Content-Type":"application/json", **_auth_headers()}, json=body)

        try:
            data = r.json()
        except Exception:
            return f"❌ Error: {r.status_code} - {r.text}"
        if r.status_code >= 400:
            return f"❌ Error {r.status_code}: {data}"

        issues = data.get("issues", [])
        is_last = data.get("isLast", True)
        next_token = data.get("nextPageToken")

        lines = [f"✓ {len(issues)} resultado(s) | isLast={is_last} | nextPageToken={next_token}"]
        for it in issues:
            key = it.get("key")
            flds = it.get("fields", {}) or {}
            summ = flds.get("summary")
            stat = (flds.get("status") or {}).get("name")
            ass  = flds.get("assignee") or {}
            assn = ass.get("displayName") if isinstance(ass, dict) else ass
            labels = flds.get("labels", [])
            lines.append(f"- {key}: {summ} | {stat} | {assn} | labels={labels}")
        return "\n".join(lines)

    def _add_comment(self, issueKey: str, comment: str) -> str:
        url = f"{_base()}/rest/api/3/issue/{issueKey}/comment"
        r = requests.post(url, json={"body": to_adf(comment)},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        return f"✓ Comentario agregado a {issueKey}"

    def _list_transitions(self, issueKey: str) -> str:
        url = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
        r = requests.get(url, headers={"Accept":"application/json", **_auth_headers()})
        r.raise_for_status()
        trans = r.json().get("transitions", [])
        if not trans:
            return "No hay transiciones disponibles (revisa permisos/flujo)."
        lines = ["Transiciones disponibles:"]
        for t in trans:
            lines.append(f"- {t.get('name')} (id={t.get('id')}) → destino: {(t.get('to') or {}).get('name')}")
        return "\n".join(lines)

    def _transition_issue(self, issueKey: str, transitionId: Optional[str] = None,
                          transitionName: Optional[str] = None,
                          fields: Optional[Dict[str,object]] = None) -> str:
        # Resolver ID por nombre si no se proporciona
        if not transitionId and transitionName:
            url_list = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
            rlist = requests.get(url_list, headers={"Accept":"application/json", **_auth_headers()})
            rlist.raise_for_status()
            opts = rlist.json().get("transitions", [])
            mapping = {t.get("name","").strip().lower(): t.get("id") for t in opts}
            transitionId = mapping.get((transitionName or "").strip().lower())
            if not transitionId:
                nombres = [t.get("name") for t in opts]
                raise ValueError(f"No encontré transición por nombre='{transitionName}'. Disponibles: {nombres}")

        if not transitionId:
            raise ValueError("Debes pasar 'transitionId' o 'transitionName'.")

        payload = {"transition": {"id": str(transitionId)}}
        if fields:
            payload["fields"] = fields  # p.ej. {"resolution": {"name": "Done"}}

        url = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
        r = requests.post(url, json=payload,
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        return f"✓ Issue {issueKey} transicionado con transitionId={transitionId}"
    
    def _get_issue(self, issueKey: str,
                   fields: Optional[List[str]] = None,
                   expand: Optional[str] = None) -> str:
        base = _base()
        fields = fields or ["summary", "status", "assignee", "labels", "issuetype"]
        params = {"fields": ",".join(fields)}
        if expand:
            params["expand"] = expand

        url = f"{base}/rest/api/3/issue/{issueKey}"
        r = requests.get(url, headers={"Accept": "application/json", **_auth_headers()}, params=params)
        try:
            data = r.json()
        except Exception:
            return f"❌ Error: {r.status_code} - {r.text}"
        if r.status_code >= 400:
            return f"❌ Error {r.status_code}: {data}"

        key = data.get("key", issueKey)
        flds = data.get("fields", {}) or {}

        summary = flds.get("summary")
        st = flds.get("status")
        status = st.get("name") if isinstance(st, dict) else st

        ass = flds.get("assignee")
        assignee = None
        if isinstance(ass, dict):
            assignee = ass.get("displayName") or ass.get("emailAddress") or ass.get("accountId")

        labels = flds.get("labels", []) if isinstance(flds.get("labels"), list) else []

        itype = flds.get("issuetype")
        issue_type = itype.get("name") if isinstance(itype, dict) else itype

        lines = [
            f"✓ {key}",
            f"- summary : {summary}",
            f"- type    : {issue_type}",
            f"- status  : {status}",
            f"- assignee: {assignee}",
            f"- labels  : {labels}",
            f"- url     : {_base()}/browse/{key}",
        ]
        return "\n".join(lines)


# Inicializador

In [6]:
srv = ServidorJira()
print(srv.get_tools())

[{'name': 'create_issue', 'description': 'Crea un issue en Jira', 'input_schema': {'type': 'object', 'properties': {'projectKey': {'type': 'string'}, 'summary': {'type': 'string'}, 'description': {'type': 'string'}, 'issueType': {'type': 'string', 'default': 'Task'}, 'assignee': {'type': 'string', 'description': 'accountId en Cloud'}, 'labels': {'type': 'array', 'items': {'type': 'string'}}, 'customFields': {'type': 'object', 'additionalProperties': True}}, 'required': ['projectKey', 'summary']}}, {'name': 'create_subtask', 'description': 'Crea una sub-tarea bajo un issue padre', 'input_schema': {'type': 'object', 'properties': {'parentKey': {'type': 'string'}, 'summary': {'type': 'string'}, 'description': {'type': 'string'}, 'subtaskTypeName': {'type': 'string', 'default': 'Sub-task'}, 'subtaskTypeId': {'type': 'string'}, 'assignee': {'type': 'string'}, 'labels': {'type': 'array', 'items': {'type': 'string'}}, 'customFields': {'type': 'object', 'additionalProperties': True}}, 'require

# Ejemplos Manuales

### Buscar

In [6]:
print(srv.ejecutar_herramienta("search_issues", {
    "jql": "issueKey = MCIA-5",
    "fields": ["summary", "status", "assignee", "labels"]
}))


✓ 1 resultado(s) | isLast=True | nextPageToken=None
- MCIA-5: Proyecto PLN | En curso | DANIEL EDUARDO GONZALEZ ALVARADO | labels=[]


### Crear Sub-tarea

In [ ]:
print(srv.ejecutar_herramienta("create_subtask",{
    "parentKey": "MCIA-5",
    "subtaskTypeName": "Subtask",
    "summary": "Sub-tarea de prueba",
    "labels": ["subtarea", "prueba"],
    "description": "Tarea creada desde VS Code",
    "customFields": {
        "project": {"key": "MCIA"}
    }
}))

✓ Sub-tarea MCIA-7 bajo MCIA-5 → https://dg-mcia.atlassian.net/browse/MCIA-7


### Comentar

In [7]:
print(srv.ejecutar_herramienta("add_comment", {
    "issueKey": "MCIA-7",
    "comment": "Comentario de prueba 2 desde la libreta."
}))

✓ Comentario agregado a MCIA-7


### Revisar y Mover de estado

In [9]:
print(srv.ejecutar_herramienta("list_transitions", {
    "issueKey": "MCIA-7"
}))

Transiciones disponibles:
- Por hacer (id=11) → destino: Por hacer
- En curso (id=21) → destino: En curso
- En revisión (id=31) → destino: En revisión
- Finalizado (id=41) → destino: Finalizado


In [ ]:
print(srv.ejecutar_herramienta("transition_issue", {
    "issueKey": "MCIA-7",
    "transitionId": "21",
    "transitionName": "En curso"
}))

✓ Issue MCIA-7 transicionado con transitionId=21


# CLI

In [7]:
class ChatJiraAnthropic:
    """
    Chat mínimo:
      - Usa Anthropic Messages API.
      - Publica las herramientas de ServidorJira.
      - Ejecuta todos los tool_use en el turno.
    """
    def __init__(
        self,
        servidor_jira: "ServidorJira",
        api_key: Optional[str] = None,
        model: Optional[str] = None,
        system_prompt: Optional[str] = None,
        max_tokens: int = 1024,
    ):
        api_key = api_key or os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("Falta ANTHROPIC_API_KEY.")
        self.client = anthropic.Anthropic(api_key=api_key)

        if not model:
            try:
                modelos = self.client.models.list()
                disponibles = {m.id for m in modelos.data}
                candidatos = [
                    m for m in disponibles
                    if "sonnet" in m or "opus" in m or "haiku" in m
                ]
                self.model = sorted(candidatos)[-1] if candidatos else modelos.data[0].id
                print(f"[Modelo] Usando automáticamente: {self.model}")
            except Exception as e:
                print(f"[Advertencia] No se pudo listar modelos ({e}), usando por defecto 'claude-sonnet-4'")
                self.model = "claude-sonnet-4"
        else:
            self.model = model
        
        self.server = servidor_jira
        self.tools = self.server.get_tools()
        self.max_tokens = max_tokens
        self.system_prompt = system_prompt or (
            "Eres un asistente de Jira. Usa herramientas cuando sea necesario. "
            "Responde en español. Si no puedes, explica el motivo."
        )
        self.messages: List[Dict[str, Any]] = []

    def _call(self, messages: List[Dict[str, Any]]):
        return self.client.messages.create(
            model=self.model,
            system=self.system_prompt,
            messages=messages,
            tools=self.tools,
            max_tokens=self.max_tokens,
        )

    def preguntar(self, texto: str, verbose: bool = True) -> str:
        if verbose:
            print("\n" + "=" * 60)
            print(f"🤓 Usuario: {texto}")
            print("=" * 60)

        self.messages.append({"role": "user", "content": texto})
        resp = self._call(self.messages)

        # Mientras el modelo solicite herramientas
        while resp.stop_reason == "tool_use":
            # Añadimos el bloque del asistente con los tool_use
            self.messages.append({"role": "assistant", "content": resp.content})

            # Ejecutamos cada tool_use
            results_blocks: List[Dict[str, Any]] = []
            for block in resp.content:
                if getattr(block, "type", None) == "tool_use":
                    name = block.name
                    args = block.input
                    if verbose:
                        print(f"\n🔧 {name}({json.dumps(args, ensure_ascii=False)})")
                    try:
                        result = self.server.ejecutar_herramienta(name, args or {})
                        is_error = isinstance(result, str) and result.strip().startswith("❌")
                        if verbose:
                            print(("   ✗ " if is_error else "   ✓ ") + str(result))
                        results_blocks.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result),
                            "is_error": is_error
                        })
                    except Exception as e:
                        msg = f"❌ Error ejecutando {name}: {type(e).__name__}: {e}"
                        if verbose:
                            print("   ✗ " + msg)
                        results_blocks.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": msg,
                            "is_error": True
                        })

            # Devolvemos resultados al modelo
            self.messages.append({"role": "user", "content": results_blocks})
            resp = self._call(self.messages)

        # Extraemos texto final
        final_text = ""
        for block in resp.content:
            if hasattr(block, "text"):
                final_text += block.text

        if verbose:
            print("\n🤖 Asistente:", final_text.strip())
            print("=" * 60)
        # Guardamos la salida final en historial
        self.messages.append({"role": "assistant", "content": resp.content})
        return final_text.strip()

# Chat

In [8]:
def chat_interactivo():
    """Modo chat interactivo con Anthropic + ServidorJira"""
    print("\n🧠 Chat Jira - Modo Interactivo")
    print("Escribe 'salir' para terminar.\n")

    # Instancia del servidor y chatbot
    servidor = ServidorJira()
    bot = ChatJiraAnthropic(servidor_jira=servidor)
    bot.messages.clear()

    while True:
        try:
            pregunta = input("💬 Tú: ").strip()
            if pregunta.lower() in ['salir', 'exit', 'quit']:
                print("👋 ¡Hasta luego!")
                break

            if pregunta:
                bot.preguntar(pregunta, verbose=True)

        except KeyboardInterrupt:
            print("\n👋 ¡Hasta luego!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

# Ejecutar modo interactivo solo si se corre directamente
if __name__ == "__main__":
    chat_interactivo()



🧠 Chat Jira - Modo Interactivo
Escribe 'salir' para terminar.

[Modelo] Usando automáticamente: claude-sonnet-4-5-20250929

🤓 Usuario: Cual es el estado actual de key = MCIA-7

🔧 get_issue({"issueKey": "MCIA-7"})
   ✓ ✓ MCIA-7
- summary : Sub-tarea de prueba
- type    : Subtask
- status  : En curso
- assignee: DANIEL EDUARDO GONZALEZ ALVARADO
- labels  : ['prueba', 'subtarea']
- url     : https://dg-mcia.atlassian.net/browse/MCIA-7

🤖 Asistente: El estado actual del issue **MCIA-7** es:

- **Título**: Sub-tarea de prueba
- **Tipo**: Subtarea
- **Estado**: **En curso**
- **Asignado a**: DANIEL EDUARDO GONZALEZ ALVARADO
- **Etiquetas**: prueba, subtarea
- **URL**: https://dg-mcia.atlassian.net/browse/MCIA-7

¿Necesitas realizar algún cambio en este issue?

🤓 Usuario: Agregar un comentario

🤖 Asistente: ¿Qué comentario te gustaría agregar al issue MCIA-7?

🤓 Usuario: Prueba 3 desde chatbot

🔧 add_comment({"issueKey": "MCIA-7", "comment": "Prueba 3 desde chatbot"})
   ✓ ✓ Comentario agreg